<a href="https://colab.research.google.com/github/lmendezayl/uba-optimizacion-tp1/blob/main/tp1_optimizacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Redes Neuronales: Aplicacion del Metodo de Descenso por Gradiente

Para esta primera parte trabajaremos con el archivo `utils_tools.jl`, el dataset `Iris` y seguiremos los lineamientos vistos aquı. Utilizaremos una red con tres capas (con funciones de activacion ReLu, identidad y softmax) de 5 niveles.

In [2]:
import Pkg; Pkg.add("RDatasets")

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed Mocking ─────────── v0.8.1
   Installed TimeZones ───────── v1.22.2
   Installed RData ───────────── v1.1.0
   Installed TZJData ─────────── v1.5.0+2025b
   Installed CategoricalArrays ─ v1.1.0
   Installed RDatasets ───────── v0.8.1
  Installing 1 artifacts
   Installed artifact tzjdata                  112.5 KiB
    Updating `~/.julia/environments/v1.12/Project.toml`
  [ce6b1742] + RDatasets v0.8.1
    Updating `~/.julia/environments/v1.12/Manifest.toml`
  [324d7699] + CategoricalArrays v1.1.0
  [78c3b35d] + Mocking v0.8.1
  [df47a6cb] + RData v1.1.0
  [ce6b1742] + RDatasets v0.8.1
  [dc5dba14] + TZJData v1.5.0+2025b
  [f269a46b] + TimeZones v1.22.2
Precompiling packages...
   4132.4 ms  ✓ TZJData
   3522.9 ms  ✓ Mocking
   6895.3 ms  ✓ CategoricalArrays
   2680.1 ms  ✓ CategoricalArrays → CategoricalArraysJSONExt
   3919.0 ms  ✓ CategoricalArrays → CategoricalArraysStatsBaseExt


In [3]:
using LinearAlgebra, RDatasets, Random, Statistics, Plots

In [4]:
iris = dataset("datasets", "iris")
X = Matrix(iris[:, 1:4])
y = iris.Species

150-element CategoricalArrays.CategoricalArray{String,1,UInt8}:
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 ⋮
 "virginica"
 "virginica"
 "virginica"
 "virginica"
 "virginica"
 "virginica"
 "virginica"
 "virginica"
 "virginica"
 "virginica"
 "virginica"
 "virginica"

In [5]:
# utils_tools.jl
function split(X, y; dims=1, ratio_train=0.8)
    n = length(y)

    n_train = round(Int, ratio_train*n) # Redondeamos el corte
    i_rand = randperm(n)				# Permutamos los indices
    i_train = i_rand[1:n_train] 		# Usamos el 80% para train
    i_test = i_rand[n_train+1:end] 		# El resto lo usamos para test

    X_train = X[i_train,:]
    y_train = y[i_train]
    X_test  = X[i_test, :]
    y_test  = y[i_test]
    return X_train, y_train, X_test, y_test

end

function normalize(X_train, X_test; dims=1)
    col_mean = mean(X_train; dims)
    col_std = std(X_train; dims)

    return (X_train .- col_mean) ./ col_std, (X_test .- col_mean) ./ col_std
end

function onehot(y, classes)
	y_onehot = zeros(length(classes), length(y))
	num_of_class = 1:length(classes)

	for i in 1:length(y)
		y_onehot[:,i] = y[i].==classes
	end
	return y_onehot
end

function prepare_data(X, y; do_normal=true, do_onehot=true, kwargs...)
    X_train, y_train, X_test, y_test = split(X, y)

    if do_normal
        X_train, X_test = normalize(X_train, X_test; kwargs...)
    end

    classes = unique(y)

    if do_onehot
        y_train = onehot(y_train, classes)
        y_test = onehot(y_test, classes)
    end

    return X_train', y_train, X_test', y_test, classes
end

X_train, y_train, X_test, y_test, classes = prepare_data(X, y)

mean_tuple(d::AbstractArray{<:Tuple}) = Tuple([mean([d[k][i] for k in 1:length(d)]) for i in 1:length(d[1])])

grad_total(modelo,grad,x,y) = mean_tuple([grad(modelo, X_train[:,k], y_train[:,k]) for k in 1:size(X_train,2)])

# funciones de activacion
relu(x) = max.(0,x)
id(x) = x
softmax(x) = exp.(x) ./ sum(exp.(x), dims=1)

softmax (generic function with 1 method)

In [9]:
println(size(X_train, 1))
println(size(X_train, 2))
println(size(y_train, 1))
println(size(y_train, 2))

4
120
3
120


In [11]:
mutable struct RedNeuronal{T<:Real}
	# Utilizaremos una red neuronal con tres capas con funciones de activacion ReLU,
	# identidad y softmax de 5 niveles.
	#
	# Nota: no es como Python, no hace falta definir una clase, ni tapoco
	# hace falta definir atributos como __init__ para poder definir el llamado a
	# el constructor de esta clase. Aca directamente llamamos
	# m = RedNeuronal(W1, b1, W2, b2)
	# y listo, con esto es suficiente para crear el modelo.
	#
	# El downside es que no podemos crear metodos bajo el struct RedNeuronal
	W1::Matrix{T} # l1 -> l2
	b1::Vector{T}
	W2::Matrix{T} # l2 -> l3
	b2::Vector{T}
end

# creamos el modelo con el constructor de Red Neuronal
Random.seed!(73)
W1 = randn(5, size(X_train, 1)) # este 1 era un 2, tomando una matriz de 5x4 enlugar de 5x120!
b1 = randn(5)
W2 = randn(size(y_train, 1), 5)
b2 = randn(size(y_train, 1 ))

model = RedNeuronal(W1, b1, W2, b2)

RedNeuronal{Float64}([1.2123689466501288 -1.8659948470385643 1.5359646820970976 -2.946693261135319; 0.07867339151355215 -0.5409383051722126 1.904244155888974 0.4000651169550351; … ; -1.4081384368973173 0.3566138060639536 0.758845608555424 -0.2462781339520417; 0.4644730607385192 -0.4580210630616379 -2.3363570247525765 0.6032439841236702], [0.036509117320801934, 2.0183941863348958, -1.4222121770628655, -0.02458224672669713, 0.5662594627526059], [0.9865857065343743 0.5569751877519062 … -0.013964807677120253 -1.1131349472530667; 0.5499123327973198 0.12834599681305806 … -0.7764680654747667 0.36181588423766176; -0.7411473285232308 -0.06050648910140387 … -0.7658544536287244 2.1530628891736723], [-0.14226841165752968, -1.0368122047759714, -0.14131008223303249])

## Ejercicio 1
Implementar una función `forward_pass(model,x)` que para cada dato $x$ retorne la predicción según el modelo.

In [12]:
function forward_pass(model, X) # Done
	# Implementar una funcion que para cada dato x retorne la prediccion segun
	# el modelo.
	W1, b1, W2, b2 = model.W1, model.b1, model.W2, model.b2
	z_1 = W1 * X .+ b1
	size(z_1)
	a_1 = relu(z_1)
	z_2 = W2 * a_1 .+ b2
	a_2 = z_2
	y = softmax(a_2)
	return y
end

forward_pass (generic function with 1 method)

In [13]:
forward_pass(model, X_train)

3×120 Matrix{Float64}:
 0.926366   0.863228   0.848155   …  0.880232   0.910422   0.914817
 0.051925   0.0772402  0.101873      0.0573091  0.0454764  0.0478143
 0.0217092  0.0595321  0.0499716     0.0624587  0.0441012  0.0373687

## Ejercicio 2
Implementar una función `grad(model,x,y)` que calcule el gradiente de la función de pérdida para cada par de datos $(x,y)$ utilizando $\texttt{back-propagation}$. Y utilizarla para calcular el gradiente completo usando la función `grad_total` de `utils_tools.jl`.

In [14]:
# TODO
function grad(model, X, y)
	# Implementar una funcion que calcule el gradiente de la funcion de perdida para
	# cada par de datos (x,y) utilizando backprop. Usarla para calcular el grad completo
	# usando la funcion grad_total de utils_tools.jl

	W1, b1, W2, b2 = model.W1, model.b1, model.W2, model.b2

    z_1 = W1 * X .+ b1
    a_1 = relu.(z_1)
    z_2 = W2 * a_1 .+ b2
    y_pred = softmax(z_2)

	relu_prime(X) = X > 0 ? 1.0 : 0.0 # heaviside = relu'

    delta_2 = y_pred .- y # https://eli.thegreenplace.net/2016/the-softmax-function-and-its-derivative/
    grad_W2 = delta_2 * a_1'
    grad_b2 = delta_2
    delta_1 = (W2' * delta_2) .* relu_prime.(z_1) # .* es prod de hadamard
    grad_W1 = delta_1 * X'
    grad_b1 = delta_1
    return (grad_W1, grad_b1, grad_W2, grad_b2)
end

grad (generic function with 1 method)

In [15]:
grad(model, X_train, y_train)

([49.85918883522952 -25.19039049543795 51.628680147950256 42.78626088218262; 23.524462805222413 -14.927845903065375 28.12779523493365 27.461534337155246; … ; 25.866248872742712 -37.393992031571436 44.384773183293184 42.979911598337786; -149.6299191855053 157.9828252274604 -193.27403135518844 -186.80959955578612], [0.37649137957834133 0.30008912006291083 … 0.0 1.6422906679072893; 0.39296756792080334 0.35876183056559685 … 0.5707574921140236 0.57391257263687; … ; 0.0 0.0 … 0.0 0.6873339220777008; -0.0 -1.166581840845928 … -0.0 -3.073620501399879], [75.19029265155726 271.5476834920117 … -34.9878101900106 -68.25590709638045; -52.83755886447079 -102.7743127044231 … -4.912433693672018 -11.70225690557899; -22.352733787086482 -168.77337078758865 … 39.90024388368262 79.95816400195946], [0.926365758664354 0.8632277441180876 … 0.910422416428211 0.9148169664200235; -0.9480749657690357 -0.9227598118971599 … 0.04547640308520625 0.04781434649510263; 0.0217092071046818 0.05953206777907243 … -0.95589881

## Ejercicio 3
Implementar una función `train!` que tome como parámetros, el modelo a entrenar, los conjuntos de entrenamiento, el tamaño del paso y la cantidad de iteraciones máximas para entrenar la red utilizando el método de descenso por gradiente con paso constante. La función debe retornar el modelo entrenado y un vector con los valores de la función de pérdida en cada iteración.

*Opcional:* Probar distintas funciones de activación y comparar.

In [18]:
println(size(y_train))

(3, 120)


In [ ]:
# TODO
function train!(model, X_train, y_train, step=1e-2, max_iter=500)
	# La funcion debe retornar el modelo entrenado y un vector con los valores de la
	# funcion de perdida en cada iteracion
	loss_hist = []
	cross_entropy(y_pred, y_true) = -mean(sum(y_true .* log.(y_pred)))

	for i in 1:max_iter
		# begin iter
		# correr forward pass -> y
		# calcular cross entropy -> loss
		# hacer backprop -> delta_C
		# actualizar? preguntar
		# end iter
		y_pred = forward_pass(model, X_train)
		loss = cross_entropy(y_pred, y_train)
		push!(loss_hist, loss)
		grad_W1, grad_b1, grad_W2, grad_b2 = grad_total(model, grad, X_train, y_train)
		model.W1 .-= step .* grad_W1
		model.b1 .-= step .* grad_b1
		model.W2 .-= step .* grad_W2
		model.b2 .-= step .* grad_b2

		if i % 50 == 0 || i == 1
            println("Iteración $i | Loss: $loss")
        end

	end

	return model, loss_hist
end

train! (generic function with 4 methods)

In [ ]:
model, loss_hist = train!(model, X_train, y_train)

Iteración 1 | Loss: 463.70650945428633
Iteración 50 | Loss: 78.52272013290825
Iteración 100 | Loss: 58.94257058645159
Iteración 150 | Loss: 51.23690918341371
Iteración 200 | Loss: 46.58591010769122
Iteración 250 | Loss: 43.29670064116353
Iteración 300 | Loss: 40.736740366024165
Iteración 350 | Loss: 38.66412309389607
Iteración 400 | Loss: 36.94061188152832
Iteración 450 | Loss: 35.49019151217917
Iteración 500 | Loss: 34.22382880568261


(RedNeuronal{Float64}([0.893595635744483 -1.8295782105486216 1.2111011478327676 -3.1882583684432446; 0.023921717960003287 -0.5914004992213838 1.8879224716550662 0.3740252616730989; … ; -1.5857237391614545 0.6892289763663633 0.3897118452052361 -0.5848512442014095; 1.1529776937013554 -0.8969129728960527 -1.3915586986614819 1.4504995655532995], [0.29779397424352777, 2.096306324488112, -1.4892141400337167, 0.14344783555424312, 0.3527628907252843], [0.7264634583516792 -0.3704130084396342 … 0.34316323568243207 -1.1069483552428188; 0.8010016118000776 0.5756782810363087 … -0.8497747629860006 0.42472050621710206; -0.7321143593432936 0.4195494228668863 … -1.0496757994770405 2.0839716751839825], [-0.14859256094506987, -0.7779078134031805, -0.39389032431828286]), Any[463.70650945428633, 441.6221061487861, 420.23755788034873, 399.5489475816781, 379.55990305400144, 360.2895472280613, 341.67810889267497, 323.8818047811633, 306.56629204514826, 289.98112402828025  …  34.440160901168056, 34.415887656473

## Ejercicio 4
Implementar una función `results` que retorne el gráfico de los valores de la función de pérdida para cada iteración y el rendimiento de nuestro modelo tanto en los conjuntos de entrenamiento como en los de prueba, por ejemplo, calculando $\frac{\#favorables}{\#casos\_totales}$

In [ ]:
# TODO
function results()
	# Implementar una funcion que retorne un grafico de los valores de la funcion
	# de perdida para cada iteracion y el rendimiento de nuestro modelo tanto en
	# los conjuntos de entrenamiento como en los de prueba, por ejemplo calculando
	#
	# 				 # favorables
	# 		       		---------------
	# 		      		# casos totales

end

results (generic function with 1 method)

# Perfil de desempeño

## Ejercicio 5
Implementar los siguientes métodos de descenso
* Gradiente con paso constante
* Gradiente con búsqueda de línea (condición de Armijo)
* Método de Newton
* Gradiente Conjugado No-Lineal (Fletcher-Reeves) y/o Método quasi-Newton BFGS

Cada método debe retornar: Estado (Convergió, máximas iteraciones alcanzadas, no convergió), número de iteraciones realizadas y valor final de $\|\nabla f\|$.

In [ ]:
# Cada metodo debe retornar:
# 	Estado (convegio, maximas iteraciones alcanzadas, no convergio)
# 	Numero de iteraciones realizadas
# 	Valor final de ||grad(f)||
# TODO
function gradiente_const()
end


gradiente_const (generic function with 1 method)

In [ ]:
# TODO
function gradiente_armijo()
end

gradiente_armijo (generic function with 1 method)

In [ ]:
# TODO
function newton_raphson()
end

newton_raphson (generic function with 1 method)

In [ ]:
# TODO
function gradiente_conj_no_lineal()
end

gradiente_conj_no_lineal (generic function with 1 method)

In [ ]:
# TODO
function bfgs()
end

bfgs (generic function with 1 method)

## Ejercicio 6
Utilizar cada método con las funciones incluidas en el archivo `benchmark.jl`

## Ejercicio 7
Construir el Perfil de Desempeño de Dolan & Moré para la métrica de iteraciones. Graficar y realizar un breve análisis.

*Sugerencia:* El primer paso consiste en ejecutar cada uno de los algoritmos sobre cada problema y registrar el costo (iteraciones) de cada uno. Así, podemos formar la matriz $C$ de costos. Luego se puede calcular una matriz $R$ con la formula
$$r_{s,p} = \frac{c_{s,p}}{\min\{c_{j,p}: j \in S\}}$$

In [ ]:
# TODO
function dolan_more()
	# construir perfil de desempeno de dolan & more para la metrica de iteraciones.
	# graficar y realizar un breve analisis.
	# sug: el primer paso consiste en ejecutar cada uno de los algoritmos sobre cada
	# problema y registrr el costo (iteraciones) de cada uno. Asi, podemos formar la
	# matriz C de costos. Luego se puede calcular una matriz R con la formula de r_sp
	# (ver clase de perfil de desempeno)
end

dolan_more (generic function with 1 method)